# 二项分布完整教程：从概念到应用

## 📚 教学目标
1. 理解二项分布的四个条件
2. 掌握二项概率公式：$P(x) = C(n,x) \times p^x \times (1-p)^{n-x}$
3. 用 scipy.stats.binom 计算概率
4. 计算二项分布的均值和标准差
5. 用参数判断显著性

**参考书**：《基础统计学(第14版)》（Triola）第 5-2 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

## 1. 场景设定：二项分布

### 🎯 问题
生活中有很多"成功/失败"型的实验：
- 抛硬币：正面/反面
- 产品检验：合格/不合格
- 医学检验：阳性/阴性

当我们重复进行这样的实验时，"成功"的次数就服从**二项分布**。

### 📖 书中的定义
> 二项分布适用于结果包含两种可能的场景。它必须满足四个条件：
> 1. 试验次数是固定的
> 2. 试验必须是相互独立的
> 3. 每次试验的结果有且仅有两种可能
> 4. 所有试验中成功的概率都相等

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import comb

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print("✅ 库导入完成")

## 2. 二项分布的四个条件

### 📐 条件检查
一个过程是否服从二项分布，需要检查四个条件：

| 条件 | 说明 | 检查要点 |
|------|------|----------|
| 1. 固定试验次数 | n 是预先确定的 | 不能边做边决定 |
| 2. 独立性 | 每次试验互不影响 | 或用 5% 法则 |
| 3. 二元结果 | 只有成功/失败 | 明确定义"成功" |
| 4. 等概率 | p 对每次试验相同 | 不能变来变去 |

### 📖 书中的例子
> 随机选取 10 个成年人，求正好有 2 人没有携带现金的概率。
> 检查：(1) n=10 固定 ✓ (2) 独立 ✓ (3) 有/没有现金 ✓ (4) p=0.05 等概率 ✓

In [ ]:
# ========== 步骤 1: 定义二项分布参数 ==========

# 场景：随机选取 10 个成年人，每人不带现金的概率为 0.05
n = 10   # 试验次数
p = 0.05  # 成功概率（不带现金）
q = 1 - p  # 失败概率（带现金）

print(f"\n📊 步骤 1: 定义二项分布参数")
print(f"  试验次数 n = {n}")
print(f"  成功概率 p = {p}（不带现金）")
print(f"  失败概率 q = 1 - p = {q}（带现金）")

# 检查四个条件
print(f"\n🔬 检查四个条件:")
print(f"  1. 固定试验次数: n = {n} ✓")
print(f"  2. 独立性: 随机选取，相互独立 ✓")
print(f"  3. 二元结果: 有现金/没有现金 ✓")
print(f"  4. 等概率: p = {p} 对所有人相同 ✓")
print(f"  → 满足二项分布条件！")

## 3. 二项概率公式

### 📐 计算公式
在二项分布中，正好有 $x$ 次成功的概率：
$$P(x) = C(n,x) \times p^x \times q^{n-x} = \frac{n!}{(n-x)! \times x!} \times p^x \times (1-p)^{n-x}$$

其中：
- $n$ = 固定的试验次数
- $x$ = 成功的次数
- $p$ = 一次成功的概率
- $q = 1-p$ = 一次失败的概率

### 📖 书中的例子
> 求正好有 2 人没有携带现金的概率：$P(2) = C(10,2) \times 0.05^2 \times 0.95^8 = 0.0746$

In [ ]:
# ========== 步骤 2: 手算二项概率 ==========

x = 2  # 我们想知道正好 2 人不带现金的概率

# 计算三个因子
# 因子1: C(n,x)
comb_nx = comb(n, x, exact=True)
# 因子2: p^x
p_power = p ** x
# 因子3: q^(n-x)
q_power = q ** (n - x)

# 手算概率
prob_manual = comb_nx * p_power * q_power

print(f"\n📊 步骤 2: 手算二项概率 P(x={x})")
print(f"  计算公式: P(x) = C(n,x) × p^x × q^(n-x)")
print(f"\n  计算过程:")
print(f"    因子1: C({n},{x}) = {comb_nx}")
print(f"    因子2: p^{x} = {p}^{x} = {p_power:.6f}")
print(f"    因子3: q^({n}-{x}) = {q}^{n-x} = {q_power:.6f}")
print(f"\n  P({x}) = {comb_nx} × {p_power:.6f} × {q_power:.6f}")
print(f"  P({x}) = {prob_manual:.6f}")
print(f"\n💡 正好有 {x} 人不带现金的概率 ≈ {prob_manual:.4f}（约 {prob_manual*100:.2f}%）")

In [ ]:
# ========== 步骤 3: 用 scipy.stats.binom 验证 ==========

# 使用 scipy 计算
prob_scipy = stats.binom.pmf(x, n, p)

print(f"\n📊 步骤 3: 用 scipy.stats.binom 验证")
print(f"  手算 P({x}) = {prob_manual:.6f}")
print(f"  scipy P({x}) = {prob_scipy:.6f}")
print(f"\n  ✅ 验证通过！")

## 4. 完整的概率分布

### 📐 概念
我们可以计算 $x = 0, 1, 2, \ldots, n$ 的所有概率，得到完整的二项概率分布。

### 📖 书中的例子
> 当 $n=10, p=0.05$ 时，Excel 给出了所有 $x$ 值对应的概率。

In [ ]:
# ========== 步骤 4: 完整的二项概率分布 ==========

# 计算所有 x 值的概率
x_all = np.arange(0, n + 1)
probs_all = stats.binom.pmf(x_all, n, p)

# 创建 DataFrame
df_binom = pd.DataFrame({
    'x': x_all,
    'P(x)': probs_all,
    'P(x) 百分比': [f'{p*100:.4f}%' for p in probs_all]
})

print(f"\n📊 步骤 4: 完整的二项概率分布 (n={n}, p={p})")
print(df_binom.to_string(index=False))

# 验证概率之和
print(f"\n🔬 验证: ΣP(x) = {probs_all.sum():.6f}")

In [ ]:
# ========== 可视化: 二项概率分布 ==========

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(x_all, probs_all, width=0.6, color='steelblue', edgecolor='white', alpha=0.8)

# 高亮 x=2
bars[x].set_color('#e74c3c')
bars[x].set_alpha(1.0)

# 在柱子上标注概率
for i, (bar, prob) in enumerate(zip(bars, probs_all)):
    if prob > 0.001:  # 只标注大于 0.1% 的概率
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{prob:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('x (Number of People Without Cash)', fontsize=12)
ax.set_ylabel('P(x) (Probability)', fontsize=12)
ax.set_title(f'Binomial Distribution: n={n}, p={p}', fontsize=14, fontweight='bold')
ax.set_xticks(x_all)
ax.grid(axis='y', alpha=0.3)

# 添加注释
ax.annotate(f'P({x}) = {probs_all[x]:.4f}',
            xy=(x, probs_all[x]), xytext=(x + 1.5, probs_all[x] + 0.05),
            arrowprops=dict(arrowstyle='->', color='#e74c3c'),
            fontsize=11, color='#e74c3c', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  红色柱子 = P(x={x}) = {probs_all[x]:.4f}")
print(f"  最可能的结果是 0 人不带现金（概率 {probs_all[0]:.4f}）")
print(f"  分布严重右偏，因为 p=0.05 很小")

## 5. 二项分布的均值和标准差

### 📐 计算公式
**均值**：
$$\mu = n \times p$$

**标准差**：
$$\sigma = \sqrt{n \times p \times q}$$

**方差**：
$$\sigma^2 = n \times p \times q$$

### 📖 书中的例子
> $n=10, p=0.05$ 时，$\mu = 10 \times 0.05 = 0.5$，$\sigma = \sqrt{10 \times 0.05 \times 0.95} = 0.6892$

In [ ]:
# ========== 步骤 5: 计算均值和标准差 ==========

# 方法1：使用二项分布公式
mean_binom = n * p
var_binom = n * p * q
std_binom = np.sqrt(var_binom)

# 方法2：使用概率分布定义（验证）
mean_def = np.sum(x_all * probs_all)
var_def = np.sum((x_all - mean_def)**2 * probs_all)
std_def = np.sqrt(var_def)

print(f"\n📊 步骤 5: 计算均值和标准差")
print(f"\n  方法1: 二项分布公式")
print(f"    均值 μ = n × p = {n} × {p} = {mean_binom:.4f}")
print(f"    方差 σ² = n × p × q = {n} × {p} × {q} = {var_binom:.4f}")
print(f"    标准差 σ = √(n × p × q) = √{var_binom:.4f} = {std_binom:.4f}")

print(f"\n  方法2: 概率分布定义（验证）")
print(f"    均值 μ = Σ[x × P(x)] = {mean_def:.4f}")
print(f"    方差 σ² = Σ[(x-μ)² × P(x)] = {var_def:.4f}")
print(f"    标准差 σ = √σ² = {std_def:.4f}")

print(f"\n🔬 两种方法验证:")
print(f"  公式法: μ = {mean_binom:.4f}, σ = {std_binom:.4f}")
print(f"  定义法: μ = {mean_def:.4f}, σ = {std_def:.4f}")
print(f"  ✅ 验证通过！")

## 6. 显著性判断

### 📐 范围经验法则
- **显著低**：$\leq \mu - 2\sigma$
- **显著高**：$\geq \mu + 2\sigma$

### 📐 概率方法
- **显著高**：$P(x \text{ 或更多}) \leq 0.05$
- **显著低**：$P(x \text{ 或更少}) \leq 0.05$

### 📖 书中的例子
> 抛硬币 460 次出现 252 次正面是否显著高？
> $P(\geq 252) = 0.0224 < 0.05$，所以是显著高的。

In [ ]:
# ========== 步骤 6: 显著性判断 ==========

# 场景：抛硬币 460 次，出现 252 次正面
n_coins = 460
p_coins = 0.5
x_observed = 252

# 计算均值和标准差
mu_coins = n_coins * p_coins
sigma_coins = np.sqrt(n_coins * p_coins * (1 - p_coins))

# 范围经验法则
low_coins = mu_coins - 2 * sigma_coins
high_coins = mu_coins + 2 * sigma_coins

print(f"\n📊 步骤 6: 显著性判断")
print(f"  场景: 抛硬币 {n_coins} 次，出现 {x_observed} 次正面")
print(f"\n  均值 μ = n × p = {n_coins} × {p_coins} = {mu_coins:.2f}")
print(f"  标准差 σ = √(n × p × q) = {sigma_coins:.2f}")
print(f"\n  范围经验法则:")
print(f"    显著低: ≤ {low_coins:.2f}")
print(f"    显著高: ≥ {high_coins:.2f}")
print(f"    观测值 {x_observed} {'≥' if x_observed >= high_coins else '<'} {high_coins:.2f}")
print(f"    → {'显著高' if x_observed >= high_coins else '非显著'}")

# 概率方法
p_252_or_more = 1 - stats.binom.cdf(x_observed - 1, n_coins, p_coins)

print(f"\n  概率方法:")
print(f"    P(x ≥ {x_observed}) = {p_252_or_more:.6f}")
print(f"    显著性水平 α = 0.05")
print(f"    {'P < α → 显著高！' if p_252_or_more < 0.05 else 'P ≥ α → 非显著'}")

print(f"\n💡 结论：")
if p_252_or_more < 0.05:
    print(f"  460 次抛硬币出现 252 次正面是显著高的！")
    print(f"  这不太可能是碰巧，说明硬币可能不公平。")
else:
    print(f"  460 次抛硬币出现 252 次正面不是显著高的。")

In [ ]:
# ========== 可视化: NFL 加时赛问题 ==========

fig, ax = plt.subplots(figsize=(12, 6))

# 绘制二项分布
x_range = np.arange(200, 270)
probs_range = stats.binom.pmf(x_range, n_coins, p_coins)

ax.bar(x_range, probs_range, width=0.8, color='steelblue', edgecolor='white', alpha=0.7, label='P(x)')

# 高亮 x>=252 的区域
x_highlight = np.arange(x_observed, 270)
probs_highlight = stats.binom.pmf(x_highlight, n_coins, p_coins)
ax.bar(x_highlight, probs_highlight, width=0.8, color='#e74c3c', edgecolor='white', alpha=0.8, label=f'P(x≥{x_observed}) = {p_252_or_more:.4f}')

# 标记均值
ax.axvline(x=mu_coins, color='green', linestyle='-', linewidth=2, label=f'Mean = {mu_coins:.0f}')

# 标记显著高阈值
ax.axvline(x=high_coins, color='orange', linestyle='--', linewidth=2, label=f'μ+2σ = {high_coins:.0f}')

ax.set_xlabel('x (Number of Heads)', fontsize=12)
ax.set_ylabel('P(x) (Probability)', fontsize=12)
ax.set_title(f'Binomial Distribution: n={n_coins}, p={p_coins} (NFL Overtime)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  蓝色 = 二项分布概率")
print(f"  红色 = P(x ≥ {x_observed}) = {p_252_or_more:.4f}")
print(f"  绿线 = 均值 ({mu_coins:.0f})")
print(f"  橙色虚线 = 显著高阈值 ({high_coins:.0f})")
print(f"  观测值 {x_observed} 在显著高阈值之上")

## 7. 扩展：不同参数的二项分布

### 🎯 问题
观察不同 $n$ 和 $p$ 值如何影响二项分布的形态。

In [ ]:
# ========== 步骤 7: 不同参数的二项分布 ==========

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = [
    (10, 0.5, 'n=10, p=0.5 (对称)'),
    (10, 0.1, 'n=10, p=0.1 (右偏)'),
    (20, 0.5, 'n=20, p=0.5 (对称)'),
    (20, 0.1, 'n=20, p=0.1 (右偏)'),
]

for ax, (n_i, p_i, title) in zip(axes.flatten(), params):
    x_i = np.arange(0, n_i + 1)
    probs_i = stats.binom.pmf(x_i, n_i, p_i)
    
    ax.bar(x_i, probs_i, width=0.6, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('P(x)', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # 标注均值
    mu_i = n_i * p_i
    ax.axvline(x=mu_i, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.text(mu_i + 0.5, max(probs_i) * 0.9, f'μ={mu_i:.1f}', color='red', fontsize=10)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  1. p=0.5 时分布对称，p≠0.5 时分布偏斜")
print(f"  2. n 越大，分布越'平坦'，越接近正态分布")
print(f"  3. 均值 μ = n × p，随 n 和 p 线性增长")

## 8. 综合应用：产品检验

### 🎯 问题
某工厂的产品不合格率为 3%。随机抽检 20 件产品，求：
1. 正好有 1 件不合格的概率
2. 最多有 1 件不合格的概率
3. 至少有 2 件不合格的概率

In [ ]:
# ========== 步骤 8: 综合应用 — 产品检验 ==========

# 参数
n_inspect = 20
p_defect = 0.03

# 计算各种概率
# 1. P(x = 1)
p_exactly_1 = stats.binom.pmf(1, n_inspect, p_defect)

# 2. P(x ≤ 1)
p_at_most_1 = stats.binom.cdf(1, n_inspect, p_defect)

# 3. P(x ≥ 2)
p_at_least_2 = 1 - stats.binom.cdf(1, n_inspect, p_defect)

# 均值和标准差
mu_inspect = n_inspect * p_defect
sigma_inspect = np.sqrt(n_inspect * p_defect * (1 - p_defect))

print(f"\n📊 步骤 8: 综合应用 — 产品检验")
print(f"  参数: n = {n_inspect}, p = {p_defect}")
print(f"  均值 μ = {n_inspect} × {p_defect} = {mu_inspect:.2f}")
print(f"  标准差 σ = √({n_inspect} × {p_defect} × {1-p_defect}) = {sigma_inspect:.4f}")

print(f"\n📊 概率计算:")
print(f"  1. P(x = 1) = {p_exactly_1:.6f}")
print(f"  2. P(x ≤ 1) = {p_at_most_1:.6f}")
print(f"  3. P(x ≥ 2) = {p_at_least_2:.6f}")

print(f"\n💡 解读:")
print(f"  - 正好 1 件不合格的概率 = {p_exactly_1*100:.2f}%")
print(f"  - 最多 1 件不合格的概率 = {p_at_most_1*100:.2f}%")
print(f"  - 至少 2 件不合格的概率 = {p_at_least_2*100:.2f}%")

# 显著性判断
high_threshold_inspect = mu_inspect + 2 * sigma_inspect
print(f"\n📊 显著性判断:")
print(f"  显著高: ≥ {high_threshold_inspect:.2f}")
print(f"  即如果发现 {int(np.ceil(high_threshold_inspect))} 件或更多不合格，就是显著高")

In [ ]:
# ========== 可视化: 产品检验概率分布 ==========

fig, ax = plt.subplots(figsize=(10, 6))

x_inspect = np.arange(0, 6)
probs_inspect = stats.binom.pmf(x_inspect, n_inspect, p_defect)

bars = ax.bar(x_inspect, probs_inspect, width=0.6, color='steelblue', edgecolor='white', alpha=0.8)

# 高亮 x=0 和 x=1
bars[0].set_color('#2ecc71')
bars[1].set_color('#2ecc71')

# 在柱子上标注概率
for bar, prob in zip(bars, probs_inspect):
    if prob > 0.001:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{prob:.4f}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('x (Number of Defective Items)', fontsize=12)
ax.set_ylabel('P(x) (Probability)', fontsize=12)
ax.set_title(f'Binomial Distribution: n={n_inspect}, p={p_defect} (Quality Inspection)', fontsize=14, fontweight='bold')
ax.set_xticks(x_inspect)
ax.grid(axis='y', alpha=0.3)

# 添加注释
ax.annotate(f'P(x≤1) = {p_at_most_1:.4f}',
            xy=(0.5, max(probs_inspect) * 0.8),
            fontsize=12, color='#2ecc71', fontweight='bold',
            ha='center')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  绿色柱子 = P(x ≤ 1) = {p_at_most_1:.4f}")
print(f"  最可能的结果是 0 件不合格（概率 {probs_inspect[0]:.4f}）")
print(f"  因为不合格率 p=0.03 很小，分布严重右偏")

## 📌 核心概念回顾

### 📌 二项分布的四个条件
- **固定试验次数**: n 是预先确定的
- **独立性**: 每次试验互不影响（或 n ≤ 5% N）
- **二元结果**: 只有成功/失败
- **等概率**: p 对每次试验相同

### 📌 二项概率公式
- **公式**: $P(x) = C(n,x) \times p^x \times (1-p)^{n-x}$
- **含义**: 正好有 x 次成功的概率
- **计算**: 三个因子相乘

### 📌 均值和标准差
- **均值**: $\mu = np$
- **标准差**: $\sigma = \sqrt{npq}$
- **含义**: 期望的成功次数和波动范围

### 📌 显著性判断
- **范围经验法则**: $\mu \pm 2\sigma$
- **概率方法**: $P(x \text{ 或更多/更少}) \leq 0.05$
- **含义**: 判断观测结果是否异常

### 🔗 完整流程
```
检查条件 → 确定参数 → 计算概率 → 计算均值/标准差 → 判断显著性
    ↓          ↓          ↓            ↓              ↓
  4个条件    n, p, q    P(x)=C×p×q    μ=np, σ=√npq    μ±2σ
```

### 📝 检验指标汇总

| 指标 | 公式 | 含义 |
|------|------|------|
| 概率 P(x) | $C(n,x) p^x q^{n-x}$ | 正好 x 次成功的概率 |
| 均值 μ | $np$ | 期望的成功次数 |
| 标准差 σ | $\sqrt{npq}$ | 成功次数的波动范围 |
| 显著低 | $\leq \mu - 2\sigma$ | 异常少的成功 |
| 显著高 | $\geq \mu + 2\sigma$ | 异常多的成功 |

## ❌ 常见误区

### ❌ 误区 1: 不检查四个条件就使用二项分布
**✓ 正确理解**: 必须满足四个条件才能使用二项分布。特别是独立性条件，如果抽样超过总体的 5%，需要用超几何分布。

### ❌ 误区 2: 混淆 P(x=a) 和 P(x≤a)
**✓ 正确理解**: $P(x=a)$ 是正好 a 次成功的概率，$P(x≤a)$ 是最多 a 次成功的概率（累积概率）。两者完全不同！

### ❌ 误区 3: 忘记 q = 1-p
在二项概率公式中，$q$ 不是一个独立参数，它等于 $1-p$。如果 $p=0.05$，则 $q=0.95$。

### ❌ 误区 4: 认为二项分布总是对称的
**✓ 正确理解**: 只有当 $p=0.5$ 时，二项分布才对称。当 $p<0.5$ 时右偏，当 $p>0.5$ 时左偏。

### ❌ 误区 5: 在不满足参数检验条件时误用参数方法
**✓ 正确理解**: 二项分布是离散分布，当 n 很大且 p 接近 0.5 时，可以用正态分布近似。但当 n 很小或 p 接近 0 或 1 时，必须使用精确的二项分布。